# Учебное исследование города Приморск (Выборгский район, Ленинградская область, Россия) с помощью BlocksNet 1.0.0.a9

**Цель:** подготовить данные, выделить городские кварталы, загрузить пешеходный граф дорожной сети и посчитать *все* показатели из `examples/pipeline.ipynb` для города Приморск.

**Важные ограничения:**
- Рабочая среда: Windows. Все команды ниже даны под PowerShell.
- Версия библиотеки: `blocksnet == 1.0.0a9` (установлена в окружении).
- Только пешеходный граф (`network_type='walk'`), без интермодального расчёта.
- Скачивание данных, расчёты и построение графа **выполняешь самостоятельно** при запуске ячеек ноутбука. Я ничего не скачивал и не запускал.
- Используется **только** граница города Приморск (не всё Приморское городское поселение).
- Результаты сохраняются в `GeoJSON`. Весь промежуточный результат можно переиспользовать без повторного скачивания.
- Расчёт времени в пути выполняется по **только одному** пешеходному графу, время переводится в **минуты**.

## 1. Подготовка окружения

В этом блоке - только проверка. **Ничего не устанавливаем.** Опираемся на существующий `pyproject.toml` текущего проекта.

**Проверка зависимостей из pyproject.toml:**
- В `dependencies` текущего `pyproject.toml` блок `blocksnet` отсутствует. Это нужно будет добавить в дальнейшем при подготовке окружения под публикацию/развёртывание. **Второй файл зависимостей создавать не будем**.
- Для работы по пайплайну потребуются: `blocksnet==1.0.0a9`, `geopandas`, `osmnx`, `networkx`, `shapely`, `pyproj`, `matplotlib` (для быстрой визуальной проверки).

In [ ]:
# Проверка версий установленных библиотек (только чтение)
import sys
print('Python:', sys.version)

try:
    import blocksnet
    print('blocksnet:', blocksnet.__version__)
except Exception as e:
    print('blocksnet: НЕ НАЙДЕН -', e)

try:
    import geopandas as gpd
    print('geopandas:', gpd.__version__)
except Exception as e:
    print('geopandas: НЕ НАЙДЕН -', e)

try:
    import osmnx as ox
    print('osmnx:', ox.__version__)
except Exception as e:
    print('osmnx: НЕ НАЙДЕН -', e)

try:
    import networkx as nx
    print('networkx:', nx.__version__)
except Exception as e:
    print('networkx: НЕ НАЙДЕН -', e)

try:
    import shapely
    print('shapely:', shapely.__version__)
except Exception as e:
    print('shapely: НЕ НАЙДЕН -', e)

### Команды для Windows (PowerShell) для работы с ноутбуком

Выполняй по мере надобности **в терминале PowerShell** из корня проекта:

```powershell
# Активировать venv, если используется
.\.venv\Scripts\Activate.ps1

# Запуск Jupyter Lab
jupyter lab

# Или Jupyter Notebook
jupyter notebook
```

**Порядок работы:** запускай ячейки по порядку. **После раздела 3** будет явная остановка - не переходи к расчёту показателей, пока не задумаешься о доступных метриках (как указано ниже).

## 2. Структура папок для данных

Создадим папку для всех исходных, промежуточных и итоговых файлов. Будем проверять наличие файлов и при повторном запуске использовать уже сохранённые.

Путь: `data/primorsk_blocksnet/`

In [ ]:
from pathlib import Path

DATA_DIR = Path('data/primorsk_blocksnet')
DATA_DIR.mkdir(parents=True, exist_ok=True)
print('Папка данных:', DATA_DIR.resolve())

# Пути под сохранение
PATH_BOUNDARY = DATA_DIR / 'boundary_primorsk.geojson'
PATH_WATER = DATA_DIR / 'water.geojson'
PATH_ROADS = DATA_DIR / 'roads.geojson'
PATH_RAILWAYS = DATA_DIR / 'railways.geojson'
PATH_BLOCKS = DATA_DIR / 'blocks.geojson'
PATH_GRAPH_GraphML = DATA_DIR / 'graph.graphml'
PATH_INDICATORS = DATA_DIR / 'indicators.geojson'

print('Готовые пути настроены.')

## 3. Загрузка границ и исходных слоёв. Подготовка кварталов

Здесь мы получаем **только границу города Приморск** (Выборгский район, Ленинградская область, Россия). Важно: не Приморское городское поселение целиком, а именно город Приморск.

Будем использовать подход из `examples/pipeline.ipynb` с `osmnx.geocode_to_gdf`, фильтровать по типам геометрии, приводить к UTM CRS, предобрабатывать и нарезать кварталы через `blocksnet.blocks.cutting`.

### 3.1. Получение границы города Приморск (с визуальной проверкой)

**Пояснения простыми словами:**
- Сначала пробуем точный запрос: `"Приморск, Выборгский район, Ленинградская область, Россия"` или вариант `"город Приморск"`. 
- Далее **обязательно** делаем визуальную проверку границы на карте, чтобы убедиться, что это именно город, а не всё поселение.
- Если при проверке граница окажется слишком большой (включает окрестные территории поселения) - вместо простого geocode лучше будет использовать более точный поиск/фильтрацию. Но для начала делаем именно так, как в примере, с проверкой.

In [ ]:
# Загрузка границы с проверкой наличия сохранённого файла
import osmnx as ox
import geopandas as gpd
import matplotlib.pyplot as plt

if PATH_BOUNDARY.exists():
    print('Найден сохранённый файл границы:', PATH_BOUNDARY)
    boundaries = gpd.read_file(PATH_BOUNDARY)
    print('Загружено из файла.')
else:
    print('Файл границы не найден. Будет выполнен запрос через osmnx.geocode_to_gdf.')
    print('Запрос: "Приморск, Выборгский район, Ленинградская область, Россия"')
    # ВАЖНО: запуск этой ячейки выполнит запрос к OSM. Ты запускаешь самостоятельно.
    boundaries = ox.geocode_to_gdf('Приморск, Выборгский район, Ленинградская область, Россия')
    # Сохраняем сразу для повторного использования
    boundaries.to_file(PATH_BOUNDARY, driver='GeoJSON')
    print('Граница сохранена в:', PATH_BOUNDARY)

# Визуальная проверка границы
ax = boundaries.plot(edgecolor='red', facecolor='none', linewidth=2)
ax.set_title('Проверка границы: город Приморск (должен быть именно город)')
ax.set_axis_off()
plt.show()

print(boundaries)

### 3.2. Загрузка дорог, ж/д, водных объектов по тегам (как в pipeline)

Используем теги из примера BlocksNet. Сохраняем слои в GeoJSON для повторного использования.

In [ ]:
bc_tags = {
    'roads': {
      "highway": ["construction","crossing","living_street","motorway","motorway_link","motorway_junction","pedestrian","primary","primary_link","raceway","residential","road","secondary","secondary_link","services","tertiary","tertiary_link","track","trunk","trunk_link","turning_circle","turning_loop","unclassified"],
      "service": ["living_street", "emergency_access"]
    },
    'railways': {
      "railway": "rail"
    },
    'water': {
      'riverbank':True,
      'reservoir':True,
      'basin':True,
      'dock':True,
      'canal':True,
      'pond':True,
      'natural':['water','bay'],
      'waterway':['river','canal','ditch'],
      'landuse':'basin',
      'water': 'lake'
    }
}

# Загрузка с проверкой файлов
def load_or_fetch_features(polygon_gdf, tags, save_path, name):
    if save_path.exists():
        print(f'{name}: найден сохранённый файл {save_path}, загружаем...')
        gdf = gpd.read_file(save_path)
        return gdf
    else:
        print(f'{name}: файла нет. Будет загрузка через ox.features_from_polygon (запуском ячейки).')
        # union_all для совместимости
        try:
            poly = polygon_gdf.union_all()
        except Exception:
            poly = polygon_gdf.geometry.unary_union
        gdf = ox.features_from_polygon(poly, tags)
        gdf.to_file(save_path, driver='GeoJSON')
        print(f'{name}: сохранён в {save_path}')
        return gdf

# Получаем данные
water = load_or_fetch_features(boundaries, bc_tags['water'], PATH_WATER, 'water')
roads = load_or_fetch_features(boundaries, bc_tags['roads'], PATH_ROADS, 'roads')
railways = load_or_fetch_features(boundaries, bc_tags['railways'], PATH_RAILWAYS, 'railways')

print('Слои загружены.')

### 3.3. Фильтрация геометрий и приведение к CRS

Фильтруем по типам геометрий как в примере, переводим всё в UTM для корректной нарезки.

In [ ]:
# Фильтрация
water_f = water[water.geom_type.isin(['Polygon', 'MultiPolygon', 'LineString', 'MultiLineString'])].copy()
roads_f = roads[roads.geom_type.isin(['LineString', 'MultiLineString'])].copy()
railways_f = railways[railways.geom_type.isin(['LineString', 'MultiLineString'])].copy()
boundaries_f = boundaries.copy()

# CRS
crs = boundaries_f.estimate_utm_crs()
print('UTM CRS:', crs)

for gdf in [water_f, roads_f, railways_f, boundaries_f]:
    gdf.to_crs(crs, inplace=True)

print('CRS приведён.')

# Визуальная проверка береговой линии и транспортной сети
ax = boundaries_f.plot(edgecolor='black', facecolor='none')
water_f.plot(ax=ax, color='lightblue', alpha=0.6)
roads_f.plot(ax=ax, color='grey', linewidth=0.5)
railways_f.plot(ax=ax, color='brown', linewidth=0.8)
ax.set_title('Визуальная проверка: граница, вода (береговая линия/водоёмы), дороги, ж/д')
ax.set_axis_off()
plt.show()

### 3.4. Предобработка и нарезка городских кварталов

Используем `preprocess_urban_objects` и `cut_urban_blocks` из `blocksnet.blocks.cutting` как в `pipeline.ipynb`.

In [ ]:
from blocksnet.blocks.cutting import preprocess_urban_objects, cut_urban_blocks

if PATH_BLOCKS.exists():
    print('Найдены сохранённые кварталы:', PATH_BLOCKS)
    blocks = gpd.read_file(PATH_BLOCKS)
    blocks = blocks.to_crs(crs)
    print('Кварталы загружены из файла. Можно пропустить нарезку, если не нужно пересчитывать.')
else:
    print('Файл кварталов не найден. Выполняем предобработку и нарезку (запуском ячейки).')
    lines, polygons = preprocess_urban_objects(roads_f, railways_f, water_f)
    blocks = cut_urban_blocks(boundaries_f, lines, polygons)
    blocks.to_file(PATH_BLOCKS, driver='GeoJSON')
    print('Кварталы сохранены в:', PATH_BLOCKS)

# Визуальная проверка полученных кварталов
ax = boundaries_f.plot(edgecolor='black', facecolor='none')
blocks.plot(ax=ax, edgecolor='blue', facecolor='none', linewidth=0.4)
ax.set_title('Визуальная проверка полученных кварталов города Приморск')
ax.set_axis_off()
plt.show()

print('Количество кварталов:', len(blocks))
blocks.head()

# ОСТАНОВКА: ПОДГОТОВКА КВАРТАЛОВ ЗАВЕРШЕНА

**ПРАВИЛО:** Перед запуском расчётов показателей **запроси у меня список доступных метрик**. Я скажу, какие метрики доступны в текущем окружении BlocksNet 1.0.0.a9 применительно к данному пайплайну.

**Что сделано на этом этапе:**
- Проверена граница (именно город Приморск)
- Выполнена визуальная проверка границы, береговой линии/воды, дорог, ж/д
- Получены и визуально проверены городские кварталы
- Кварталы сохранены в `data/primorsk_blocksnet/blocks.geojson`
- Все промежуточные слои при желании можно переиспользовать

**Дальше:** только после моего ответа о доступных метриках переходи к разделу 4.

## 4. Граф для времени в пути (только пешеходный)

**Требования:**
- Только **один** готовый граф. **Без интермодального** расчёта.
- Способ передвижения: `walk` (пешеходный). 
- Источник: OpenStreetMap через `osmnx`.
- Сначала проверяем наличие готового графа в `data/primorsk_blocksnet/graph.graphml`. Если нет - загружаем через `osmnx.graph_from_place` для города Приморск с `network_type='walk'` и сразу сохраняем в `graph.graphml`.
- Время в пути считать по этому графу, результат переводить в **минуты**.
- Источник и способ передвижения будут явно указаны в коде и пояснениях.

**Пояснение простыми словами:** матрица доступности по пешеходному графу - время пути между кварталами/объектами по пешеходной сети OSM.

In [ ]:
import networkx as nx

print('ИСТОЧНИК ГРАФА: OpenStreetMap через osmnx')
print('СПОСОБ ПЕРЕДВИЖЕНИЯ: walk (пешеходный), ТОЛЬКО ОДИН, БЕЗ ИНТЕРМОДАЛА')
print('ВРЕМЯ БУДЕТ ПЕРЕВЕДЕНО В МИНУТЫ')

if PATH_GRAPH_GraphML.exists():
    print('Найден готовый граф:', PATH_GRAPH_GraphML)
    print('Загружаем graphml...')
    G = ox.load_graphml(PATH_GRAPH_GraphML)
    print('Граф загружен из файла.')
else:
    print('Готовый graphml не найден.')
    print('Будет выполнена загрузка через ox.graph_from_place для города Приморск с network_type="walk".')
    print('ЗАПУСК ЯЧЕЙКИ ВЫПОЛНИТ СКАЧИВАНИЕ - ДЕЙСТВУЕШЬ САМОСТОЯТЕЛЬНО.')
    # Загружаем пешеходный граф только для города Приморск
    G = ox.graph_from_place('Приморск, Выборгский район, Ленинградская область, Россия', network_type='walk')
    # Сохраняем
    ox.save_graphml(G, filepath=PATH_GRAPH_GraphML)
    print('Граф сохранён в:', PATH_GRAPH_GraphML)

print('Узлов:', len(G.nodes()))
print('Рёбер:', len(G.edges()))

### 4.1. Расчёт времени между кварталами по пешеходному графу (в минутах)

**Подход (как принято в подобных пайплайнах):**
- Берём центроиды/представительные точки кварталов (`blocks`) для привязки к графу.
- Находим ближайшие узлы графа (`ox.nearest_nodes`).
- Считаем кратчайшее время по графу с учётом скорости/времени рёбер (в зависимости от настроек osmnx/networkx в BlocksNet). Итог переводим в **минуты**.
- Расчёт выполняется **только по этому пешеходному графу**, без интермодального.

**Пояснения:** ты будешь запускать эту ячейку самостоятельно. Код ниже - шаблон с пояснениями, подкорректировать можно при необходимости под API BlocksNet 1.0.0.a9, но структура сохранится.

In [ ]:
# Расчёт времени между кварталами по пешеходному графу (только walk). Время в минутах.
# Источник: OSM через osmnx, граф сохранён/загружен выше.

import numpy as np

# Если граф уже в нужной CRS - ок. blocks приведён к UTM.
blocks_g = blocks.copy()

# Берём точки для привязки
blocks_g['centroid'] = blocks_g.geometry.centroid

# Находим ближайшие узлы (в координатах графа; osmnx обычно в WGS84, но можно проецировать по необходимости)
# Примечание: в зависимости от графа может потребоваться преобразование CRS. 
# Если граф в WGS84, а центроиды в UTM - лучше перевести центроиды в EPSG:4326.
try:
    blocks_wgs = blocks_g.set_geometry('centroid').to_crs(epsg=4326)
    # nearest_nodes принимает (X,Y) = (lon, lat)
    orig = ox.nearest_nodes(G, blocks_wgs.geometry.x, blocks_wgs.geometry.y)
    print('Узлы графа найдены для кварталов.')
except Exception as e:
    print('Внимание при поиске узлов:', e)
    print('Возможно нужно проверить CRS графа. Действуй по результату при запуске.')

# Далее матрица времени: в зависимости от BlocksNet API используются встроенные функции доступности по графу.
# В pipeline.ipynb есть блоки расчёта доступности - мы повторим их все, используя ТОЛЬКО этот пешеходный граф.
# Время конвертируем в минуты (если в секундах - делим на 60).
print('Шаблон готов. Конкретный вызов функций доступности будет из blocksnet.accessibility по согласованным метрикам.')

## 5. Расчёт ВСЕХ показателей из examples/pipeline.ipynb

**КРИТИЧНО:** переходить к этому разделу **только после** ответа о доступных метриках (раздел 3 остановка).

**Требования:**
- Включить **все** показатели, которые есть в `examples/pipeline.ipynb`.
- Включить **оба** показателя обеспеченности школами.
- Включить **обе** классификации типов застройки.
- Использовать **только** пешеходный граф из раздела 4.
- Все результаты сохранять в `GeoJSON` (в `data/primorsk_blocksnet/`).
- Предусмотреть повторное использование сохранённых результатов без пересчёта.
- Время в пути - в **минутах**.

**Пояснения простыми словами:** здесь будут воспроизведены все блоки расчёта из оригинального пайплайна, адаптированные под город Приморск, под границу именно города, под пешеходный граф.

### 5.1. Шаблон для импорта модулей BlocksNet (как в pipeline)

Ниже - пример импорта. При запуске сверяй с фактическим API `blocksnet 1.0.0a9`, который установлен у тебя.

In [ ]:
# Пример импорта (адаптируй под API при необходимости)
# from blocksnet import ...
# from blocksnet.accessibility import ...
# from blocksnet.urban import ...

print('Готово к расчёту всех показателей из pipeline.ipynb.')
print('Напомни: включить ОБА показателя школ и ОБЕ классификации типов застройки.')

### 5.2. Место для расчёта всех показателей

**ИНСТРУКЦИЯ:** после моего ответа о доступных метриках скопируй/воспроизведи **все** соответствующие блоки из `https://github.com/aimclub/blocksnet/blob/main/examples/pipeline.ipynb`, адаптировав под:
- `blocks` (кварталы Приморска)
- пешеходный граф `G` из раздела 4
- сохранение в `data/primorsk_blocksnet/`
- время в **минутах**
- **оба** показателя обеспеченности школами
- **обе** классификации типов застройки

После каждого логического блока сохраняй результат в GeoJSON с понятным именем, с проверкой наличия файла для повторного использования.

In [ ]:
# РЕЗЕРВ ДЛЯ БЛОКОВ РАСЧЁТА ВСЕХ ПОКАЗАТЕЛЕЙ ИЗ PIPELINE.IPYNB
# (добавь/скопируй ячейки после получения списка метрик)
pass

## 6. Проверка результатов, план/методика/источники и инструкции повторного запуска

Раздел для итоговой сдачи.

### 6.1. Чек-лист проверки

- [ ] **Граница**: подтверждено, что используется именно город Приморск (проведена визуальная проверка)
- [ ] **Визуализация**: проверены граница, береговая линия/водоёмы, дороги, ж/д, полученные кварталы
- [ ] **Кварталы**: созданы/загружены, сохранены в `data/primorsk_blocksnet/blocks.geojson`
- [ ] **Граф**: загружен только пешеходный (`walk`), без интермодала. Источник указан (OSM/osmnx). Сохранён в `data/primorsk_blocksnet/graph.graphml`. Время переведено в **минуты**
- [ ] **Показатели**: посчитаны **все** показатели из `examples/pipeline.ipynb`, включая **оба** показателя обеспеченности школами и **обе** классификации типов застройки
- [ ] **Сохранение**: все слои/результаты сохранены в `GeoJSON`, предусмотрено переиспользование файлов
- [ ] **Остановка**: после подготовки кварталов была соблюдена (спросил про доступные метрики перед расчётом)

### 6.2. План, основные запросы, методика, источники данных (для сдачи)

**План исследования (кратко):**
1. Подготовка окружения и проверка версий (blocksnet 1.0.0a9)
2. Получение границы города Приморск (проверка: именно город, не поселение) + визуальная проверка
3. Загрузка слоёв OSM (дороги, ж/д, вода) по тегам, фильтрация, приведение CRS, визуальная проверка
4. Предобработка и нарезка кварталов через `preprocess_urban_objects`/`cut_urban_blocks`, визуальная проверка, сохранение
5. **Остановка** после кварталов, согласование доступных метрик
6. Загрузка пешеходного графа OSM (`walk`, единственный, без интермодала), сохранение `graph.graphml`, расчёт времени между кварталами в минутах
7. Расчёт всех показателей из pipeline.ipynb (включая оба показателя школ и обе классификации застройки)
8. Сохранение всех результатов в GeoJSON, проверка и подготовка к отдельному этапу - созданию HTML-страницы с картой и дашбордом

**Основные запросы к данным:**
- Граница: `Приморск, Выборгский район, Ленинградская область, Россия` (проверка соответствия "город Приморск")
- Слои: roads/railways/water по тегам из `blocksnet/examples/pipeline.ipynb`
- Граф: `ox.graph_from_place` с `network_type='walk'`

**Методика:**
- Нарезка кварталов по линейным и полигональным препятствиям (дороги, ж/д, вода) с использованием методов BlocksNet
- Пешеходная доступность по улично-дорожной сети OSM (только walk), время в пути рассчитывается по графу, переводится в минуты
- Все индикаторы рассчитываются по подходу из оригинального `pipeline.ipynb` BlocksNet

**Источники данных:**
- OpenStreetMap через `osmnx` (границы, объекты инфраструктуры, пешеходная сеть)
- BlocksNet 1.0.0.a9, примеры: https://github.com/aimclub/blocksnet (особенно `examples/pipeline.ipynb`)
- Текущий учебный проект в репозитории

**Зависимости:**
- Основные: `blocksnet==1.0.0a9`, `geopandas`, `osmnx`, `networkx`, `shapely`, `pyproj`
- В `pyproject.toml` текущего проекта `blocksnet` отсутствует (отмечено для добавления в дальнейшем, второй файл зависимостей не создавался)

### 6.3. Инструкции повторного запуска (Windows, PowerShell)

**Подготовка окружения:**
```powershell
cd "D:\ИТМО\Курс ИИ технологии\python-package-boilerplate"
.\.venv\Scripts\Activate.ps1
jupyter lab
# или jupyter notebook
```

**Порядок запуска ячеек:**
1. Раздел 1 - проверка версий (только проверка)
2. Раздел 2 - создание `data/primorsk_blocksnet/`
3. Раздел 3.1-3.4 - граница (проверка именно города Приморск + визуализация), слои, нарезка кварталов. **Сохранить `blocks.geojson`**
4. **ОСТАНОВКА** - спросить про доступные метрики. Не идти дальше, пока не получен ответ
5. Раздел 4 - пешеходный граф `walk` (проверка `graph.graphml`, иначе загрузка и сохранение). Расчёт времени в **минутах** по единственному графу
6. Раздел 5 - **все** показатели из pipeline.ipynb (оба показателя школ, обе классификации застройки). Сохранять результаты в GeoJSON
7. Раздел 6 - проверка по чек-листу

**Повторное использование без скачивания:**
- Если есть `data/primorsk_blocksnet/boundary_primorsk.geojson`, `water.geojson`, `roads.geojson`, `railways.geojson` - ячейки загрузят их из файлов
- Если есть `blocks.geojson` - можно загрузить вместо повторной нарезки
- Если есть `graph.graphml` - граф загрузится из файла, **нового скачивания не будет**
- Если есть сохранённые результаты показателей - можно загрузить их вместо пересчёта

**Базовые советы при проблемах:**
- Если граница получилась не той (включает поселение) - проверить визуализацию в 3.1, при необходимости использовать более точный запрос
- Если не находит модули - проверить активацию venv и версии из раздела 1
- Если проблемы с CRS при поиске узлов графа - сверить CRS графа и кварталов
- Помнить: **только один граф (walk), без интермодала**, время в **минутах**

**Файл создан:** `primorsk_blocksnet_pipeline.ipynb`  
**Папка для данных:** `data/primorsk_blocksnet/` (будет создана при запуске ячейки раздела 2)  
**Статус:** только создание. Ничего не скачано, не запущено, `pyproject.toml` не изменён.  
**Следующий шаг:** ты запускаешь ячейки ноутбука самостоятельно по порядку с соблюдением остановки после раздела 3.